In [1]:
import sys
import optuna
import numpy as np
import pandas as pd

sys.path.append("..")
from feature_engineering import time_based_train_test_split, get_return, round_to_step, get_mask
from databricks_connector import get_table

pd.set_option('display.max_columns', None)
target_col = "btts"
odds_col: str = "goalNoGoal_quote_currentGG"


# features = [
#  'goalNoGoal_chance_goal',
#  'goalNoGoal_chance_goalHome',
#  'goalNoGoal_chance_goalAway',
#  'goalNoGoal_multigoal_m13',
#  'goalNoGoal_multigoal_m14',
#  'goalNoGoal_multigoal_m24',
#  'goalNoGoal_multigoal_m13Home',
#  'goalNoGoal_multigoal_m13Away',
#  'goalNoGoal_multigoal_m24Home',
#  'goalNoGoal_multigoal_m24Away',
#  'goalNoGoal_quote_realGG',
#  'goalNoGoal_quote_initialGG',
#  'goalNoGoal_quote_initialNG',
#  'goalNoGoal_quote_currentGG',
#  'goalNoGoal_quote_currentNG',
#  'goalNoGoal_quote_diffRealCurrGG',
#  'goalNoGoal_quote_diffRealCurrNG',
#  'goalNoGoal_quote_diffInitialCurrGG',
#  'goalNoGoal_quote_diffInitialCurrNG',
#  'goalNoGoal_comparison_affini',
#  'goalNoGoal_comparison_flashback',
#  'goalNoGoal_stats_avgGoalHome',
#  'goalNoGoal_stats_avgGoalTakenHome',
#  'goalNoGoal_stats_avgGoalAway',
#  'goalNoGoal_stats_avgGoalTakenAway',
#  'goalNoGoal_flashback_goal',
#  'goalNoGoal_flashback_m13',
#  'goalNoGoal_flashback_m24',
#  'goalNoGoal_flashback_m35',
#  'underOver_chance_over05HT',
#  'underOver_chance_over052HT',
#  'underOver_chance_over15HT',
#  'underOver_chance_over15',
#  'underOver_chance_over25',
#  'underOver_chance_over35',
#  'underOver_chance_over45',
#  'underOver_quote_realO',
#  'underOver_quote_initialU',
#  'underOver_quote_initialO',
#  'underOver_quote_currentU',
#  'underOver_quote_currentO',
#  'underOver_quote_diffRealCurrU',
#  'underOver_quote_diffRealCurrO',
#  'underOver_quote_diffInitialCurrU',
#  'underOver_quote_diffInitialCurrO',
#  'underOver_comparison_affini',
#  'underOver_comparison_flashback',
#  'underOver_flashback_under05HT',
#  'underOver_flashback_over05HT',
#  'underOver_flashback_under15',
#  'underOver_flashback_over15',
#  'underOver_flashback_under25',
#  'underOver_flashback_over25',
#  'underOver_flashback_under35',
#  'underOver_flashback_over35',
#  'evaluation_valScala',
#  'evaluation_valMetrica',
#  'chance1x2_quote_current1',
#  'chance1x2_quote_current2'
#  ]


features = ["underOver_chance_over15HT",
"chance1x2_quote_current1",
"chance1x2_quote_current2",
"goalNoGoal_quote_currentGG",
"goalNoGoal_chance_goal",
"goalNoGoal_flashback_goal",
"goalNoGoal_stats_avgGoalHome",
"goalNoGoal_stats_avgGoalTakenAway",
"goalNoGoal_stats_avgGoalTakenHome",
"goalNoGoal_stats_avgGoalAway"]

In [2]:
# TODO: far scegliere a Optuna se tenere o meno una variabile
# TODO: aggiungere early stopping
# TODO: aggiungere mean e std col dato aggregato settimanalmente
# TODO: aggiungere handling dei nan
# TODO: in alcuni casi ho che il valore minimo consigliato è uguale a al massimo, questo porta a maschere che annullano il df, capire come risolvere (forse considerare metriche sui df binnati)

In [5]:
# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """

df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

# Train/test split
df = df_loaded.copy()
df["return"] = df.apply(lambda x: get_return(strategy=x["btts"], odds=x["goalNoGoal_quote_currentGG"]), axis=1)

df_train, df_test = time_based_train_test_split(df=df, time_col="time", train_frac=0.8)

AttributeError: 'NoneType' object has no attribute 'schema'

In [ ]:
# Define features and relative step
feature_bins_map = {
    "underOver_chance_over15HT": 5, # 5,
    "underOver_comparison_affini": 1000, #0.5, #0.1,
    "underOver_comparison_flashback": 500, #0.5, #0.1,
    # "goalNoGoal_chance_goal": 5, # ,
    # "goalNoGoal_flashback_goal": 5, # 5,
    # "goalNoGoal_stats_avgGoalHome": 0.5, #0.1, 
    # "goalNoGoal_stats_avgGoalTakenAway": 0.5, #0.1, 
    # "goalNoGoal_stats_avgGoalTakenHome": 0.5, #0.1, 
    # "goalNoGoal_stats_avgGoalAway": 0.5, #0.1, 
    # "goalNoGoal_quote_currentGG": 0.2, #0.5, #0.1
 }

# Create the binned dataframe
df_train_binned = df_train.copy()

for feat, step in feature_bins_map.items():
    df_train_binned[feat] = [round_to_step(x, step) for x in df_train_binned[feat]]

    if isinstance(step, int):
        df_train_binned[feat] = df_train_binned[feat].astype("Int64")


for feat in feature_bins_map.keys():
    print(feat +"\n")
    print(df_train_binned[feat].sort_values(ascending=True).unique())
    print("\n")

underOver_chance_over15HT

<IntegerArray>
[15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
Length: 12, dtype: Int64


underOver_comparison_affini

<IntegerArray>
[    0,  1000,  2000,  3000,  4000,  5000,  6000,  7000,  8000,  9000, 10000,
 11000, 15000, 16000, 17000]
Length: 15, dtype: Int64


underOver_comparison_flashback

<IntegerArray>
[0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 5000, 5500]
Length: 11, dtype: Int64


goalNoGoal_chance_goal

<IntegerArray>
[30, 35, 40, 45, 50, 55, 60, 65, 70]
Length: 9, dtype: Int64


goalNoGoal_flashback_goal

<IntegerArray>
[35, 40, 45, 50, 55, 60, 65, 70, <NA>]
Length: 9, dtype: Int64


goalNoGoal_stats_avgGoalHome

[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5 5.  5.5 6. ]


goalNoGoal_stats_avgGoalTakenAway

[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  4.5 5. ]


goalNoGoal_stats_avgGoalTakenHome

[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  5. ]


goalNoGoal_stats_avgGoalAway

[0.  0.5 1.  1.5 2.  2.5 3.  3.5 4.  5. ]




In [ ]:
target_col = "return"
min_obs = 50
low_cardinality_threshold = 350


def objective(trial):
    mask = pd.Series(True, index=df_train_binned.index)

    for feat, step in feature_bins_map.items():
        s = df_train_binned[feat]

        if not pd.api.types.is_numeric_dtype(s):
            continue

        non_null = s.dropna()
        if non_null.empty:
            continue

        nunique = non_null.nunique()

        include_missing = trial.suggest_categorical(
            f"{feat}_include_missing",
            [True, False]
        )

        use_min = trial.suggest_categorical(f"{feat}_use_min", [True, False])
        use_max = trial.suggest_categorical(f"{feat}_use_max", [True, False])

        # almeno un vincolo deve essere attivo
        if not use_min and not use_max:
            raise optuna.TrialPruned()

        # ---------------------------------
        # CASO 1: numerica discreta ordinata
        # ---------------------------------
        if nunique <= low_cardinality_threshold:
            unique_vals = sorted(non_null.unique().tolist())

            min_idx = trial.suggest_int(
                f"{feat}_min_idx",
                0,
                len(unique_vals) - 1
            )
            max_idx = trial.suggest_int(
                f"{feat}_max_idx",
                min_idx,
                len(unique_vals) - 1
            )

            feat_min = unique_vals[min_idx]
            feat_max = unique_vals[max_idx]

        # ---------------------------------
        # CASO 2: numerica continua/intera
        # ---------------------------------
        else:
            if pd.api.types.is_integer_dtype(s):
                feat_min = trial.suggest_int(
                    f"{feat}_min",
                    int(non_null.min()),
                    int(non_null.max()),
                    step=step
                )
                feat_max = trial.suggest_int(
                    f"{feat}_max",
                    feat_min,
                    int(non_null.max()),
                    step=step
                )
            else:
                feat_min = trial.suggest_float(
                    f"{feat}_min",
                    float(non_null.min()),
                    float(non_null.max()),
                    step=step
                )
                feat_max = trial.suggest_float(
                    f"{feat}_max",
                    feat_min,
                    float(non_null.max()),
                    step=step
                )

        # costruzione maschera feature
        feat_mask = pd.Series(True, index=s.index)

        if use_min:
            feat_mask &= s >= feat_min

        if use_max:
            feat_mask &= s <= feat_max

        if include_missing:
            feat_mask = feat_mask | s.isna()
        else:
            feat_mask = feat_mask & s.notna()

        mask &= feat_mask

    selected = df_train_binned.loc[mask]

    if len(selected) < min_obs:
        return -1e9

    mean_return = selected[target_col].mean()
    score = mean_return * np.sqrt(len(selected))

    return float(score)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-03-28 10:01:47,315] A new study created in memory with name: no-name-49103ca2-530a-443a-8a99-259dbb84bae9
[W 2026-03-28 10:01:47,332] Trial 0 failed with parameters: {} because of the following error: NameError("name 'df_train_binned' is not defined").
Traceback (most recent call last):
  File "/home/maicolnicolini/code/bet-master-analytics/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1006/1941273760.py", line 7, in objective
    mask = pd.Series(True, index=df_train_binned.index)
                                 ^^^^^^^^^^^^^^^
NameError: name 'df_train_binned' is not defined
[W 2026-03-28 10:01:47,340] Trial 0 failed with value None.


NameError: name 'df_train_binned' is not defined

In [9]:
# Create the test binned dataframe
df_test_binned = df_test.copy()

for feat, step in feature_bins_map.items():
    df_test_binned[feat] = [round_to_step(x, step) for x in df_test_binned[feat]]

    if isinstance(step, int):
        df_test_binned[feat] = df_test_binned[feat].astype("Int64")

# Get the resulting dataframes
params_dict =study.best_params



train_mask = get_mask(df_train_binned, params_dict)
test_mask = get_mask(df_test_binned, params_dict)

df_train_filtered = df_train_binned[train_mask]
df_test_filtered = df_test_binned[test_mask]

In [10]:
mean_roi = df_train_filtered["return"].mean()
right = df_train_filtered["btts"].sum()  
total = df_train_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: 0.011293103448275851
Accuracy: 0.5 (60/116)


In [11]:
mean_roi = df_test_filtered["return"].mean()
right = df_test_filtered["btts"].sum()  
total = df_test_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: -0.13606557377049178
Accuracy: 0.4 (27/61)
